In [1]:
import time
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *

spark = (SparkSession.builder
    .appName("Pedali-Part3")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.executor.memory", "1g")
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .getOrCreate())
sc = spark.sparkContext
print(spark.version, sc.defaultParallelism)

3.5.3 2


In [2]:
trips_schema = StructType([
    StructField("trip_id", StringType()), StructField("user_id", StringType()),
    StructField("vehicle_id", StringType()), StructField("vehicle_type", StringType()),
    StructField("start_station", StringType()), StructField("end_station", StringType()),
    StructField("city", StringType()), StructField("status", StringType()),
    StructField("duration_s", IntegerType()), StructField("distance_km", DoubleType()),
    StructField("battery_pct", IntegerType()), StructField("price_mad", DoubleType()),
    StructField("event_time", TimestampType()),
])
stations_schema = StructType([
    StructField("station_id", StringType()), StructField("name", StringType()),
    StructField("city", StringType()), StructField("district", StringType()),
    StructField("zone_type", StringType()), StructField("capacity", IntegerType()),
    StructField("latitude", DoubleType()), StructField("longitude", DoubleType()),
])

trips = spark.read.schema(trips_schema).json("file:///data/trips_history.jsonl")
stations = spark.read.schema(stations_schema).json("file:///data/stations.json")
trips.printSchema()
print(trips.count(), stations.count())   # attendu : 50000 et 400

root
 |-- trip_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- vehicle_id: string (nullable = true)
 |-- vehicle_type: string (nullable = true)
 |-- start_station: string (nullable = true)
 |-- end_station: string (nullable = true)
 |-- city: string (nullable = true)
 |-- status: string (nullable = true)
 |-- duration_s: integer (nullable = true)
 |-- distance_km: double (nullable = true)
 |-- battery_pct: integer (nullable = true)
 |-- price_mad: double (nullable = true)
 |-- event_time: timestamp (nullable = true)

50000 400


In [3]:
ended = trips.filter(F.col("status") == "ENDED")

ca_df = (ended.groupBy("city")
              .agg(F.round(F.sum("price_mad"), 2).alias("ca_mad"))
              .orderBy("city"))
ca_df.show()

trips.createOrReplaceTempView("trips")
spark.sql("""
  SELECT city, ROUND(SUM(price_mad), 2) AS ca_mad
  FROM trips WHERE status = 'ENDED'
  GROUP BY city ORDER BY city
""").show()

+----------+--------+
|      city|  ca_mad|
+----------+--------+
|Casablanca| 98655.9|
| Marrakech|53879.35|
|     Rabat|55483.53|
|    Tanger|36678.65|
+----------+--------+

+----------+--------+
|      city|  ca_mad|
+----------+--------+
|Casablanca| 98655.9|
| Marrakech|53879.35|
|     Rabat|55483.53|
|    Tanger|36678.65|
+----------+--------+



In [4]:
st = stations.select(F.col("station_id").alias("sid"), "zone_type", "district")
joined = ended.join(st, ended.start_station == st.sid, "inner")

by_zone = (joined.groupBy("city", "zone_type")
                 .agg(F.count("*").alias("nb_trajets"),
                      F.round(F.sum("price_mad"), 2).alias("ca_mad"))
                 .orderBy("city", F.desc("nb_trajets")))
by_zone.show(40, truncate=False)

+----------+---------------+----------+--------+
|city      |zone_type      |nb_trajets|ca_mad  |
+----------+---------------+----------+--------+
|Casablanca|residentiel    |2593      |41688.38|
|Casablanca|centre_affaires|1313      |21327.94|
|Casablanca|tourisme       |926       |14950.15|
|Casablanca|universite     |522       |8311.63 |
|Casablanca|plage          |429       |6653.57 |
|Casablanca|gare           |357       |5724.23 |
|Marrakech |residentiel    |1300      |21161.69|
|Marrakech |centre_affaires|1026      |16778.72|
|Marrakech |universite     |379       |6055.13 |
|Marrakech |tourisme       |280       |4539.28 |
|Marrakech |gare           |239       |3979.32 |
|Marrakech |plage          |83        |1365.21 |
|Rabat     |residentiel    |1170      |18477.81|
|Rabat     |centre_affaires|937       |15313.69|
|Rabat     |universite     |519       |8552.76 |
|Rabat     |tourisme       |347       |5397.03 |
|Rabat     |plage          |248       |4010.83 |
|Rabat     |gare    

In [5]:
def _band(d):
    if d is None: return None
    if d < 600:   return "court"    # < 10 min
    if d <= 1800: return "moyen"    # 10 à 30 min
    return "long"                   # > 30 min

trip_band_udf = F.udf(_band, StringType())

def band_native(c):
    return (F.when(c.isNull(), F.lit(None))
             .when(c < 600, "court")
             .when(c <= 1800, "moyen")
             .otherwise("long"))

ended.cache().count()   # on charge en mémoire pour comparer à armes égales

def timeit(label, df, n=3):
    ts = []
    for _ in range(n):
        t = time.time(); df.groupBy("band").count().collect(); ts.append(round(time.time() - t, 2))
    print(label, ts)

timeit("UDF Python    ", ended.withColumn("band", trip_band_udf("duration_s")))
timeit("when/otherwise", ended.withColumn("band", band_native(F.col("duration_s"))))

ended.withColumn("band", trip_band_udf("duration_s")).explain()
ended.withColumn("band", band_native(F.col("duration_s"))).explain()

UDF Python     [2.78, 1.11, 1.16]
when/otherwise [0.63, 0.41, 0.6]
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [trip_id#0, user_id#1, vehicle_id#2, vehicle_type#3, start_station#4, end_station#5, city#6, status#7, duration_s#8, distance_km#9, battery_pct#10, price_mad#11, event_time#12, pythonUDF0#2889 AS band#2744]
   +- BatchEvalPython [_band(duration_s#8)#2743], [pythonUDF0#2889]
      +- InMemoryTableScan [trip_id#0, user_id#1, vehicle_id#2, vehicle_type#3, start_station#4, end_station#5, city#6, status#7, duration_s#8, distance_km#9, battery_pct#10, price_mad#11, event_time#12]
            +- InMemoryRelation [trip_id#0, user_id#1, vehicle_id#2, vehicle_type#3, start_station#4, end_station#5, city#6, status#7, duration_s#8, distance_km#9, battery_pct#10, price_mad#11, event_time#12], StorageLevel(disk, memory, deserialized, 1 replicas)
                  +- *(1) Filter (isnotnull(status#7) AND (status#7 = ENDED))
                     +- FileScan json [trip_id

In [7]:
enriched = joined.withColumn("trip_band", band_native(F.col("duration_s"))).drop("sid")
silver = enriched.dropDuplicates(["trip_id", "status"])
print("avant dédoublonnage :", enriched.count(), "| après :", silver.count())

(silver.write.mode("overwrite")
       .partitionBy("city")
       .parquet("s3a://lake/silver/trips_history/"))

back = spark.read.parquet("s3a://lake/silver/trips_history/")
back.groupBy("city").count().show()

avant dédoublonnage : 15160 | après : 14851
+----------+-----+
|      city|count|
+----------+-----+
|Casablanca| 6013|
| Marrakech| 3236|
|     Rabat| 3375|
|    Tanger| 2227|
+----------+-----+



In [8]:
joined.explain(True)

== Parsed Logical Plan ==
Join Inner, (start_station#4 = sid#127)
:- Filter (status#7 = ENDED)
:  +- Relation [trip_id#0,user_id#1,vehicle_id#2,vehicle_type#3,start_station#4,end_station#5,city#6,status#7,duration_s#8,distance_km#9,battery_pct#10,price_mad#11,event_time#12] json
+- Project [station_id#26 AS sid#127, zone_type#30, district#29]
   +- Relation [station_id#26,name#27,city#28,district#29,zone_type#30,capacity#31,latitude#32,longitude#33] json

== Analyzed Logical Plan ==
trip_id: string, user_id: string, vehicle_id: string, vehicle_type: string, start_station: string, end_station: string, city: string, status: string, duration_s: int, distance_km: double, battery_pct: int, price_mad: double, event_time: timestamp, sid: string, zone_type: string, district: string
Join Inner, (start_station#4 = sid#127)
:- Filter (status#7 = ENDED)
:  +- Relation [trip_id#0,user_id#1,vehicle_id#2,vehicle_type#3,start_station#4,end_station#5,city#6,status#7,duration_s#8,distance_km#9,battery_p

In [9]:
 t = time.time()
res = (spark.read.schema(trips_schema).json("file:///data/trips_history.jsonl")
         .filter(F.col("status") == "ENDED")
         .groupBy("city").agg(F.sum("price_mad")).collect())
print("Pipeline DataFrame (CA par ville) : %.2f s" % (time.time() - t))

Pipeline DataFrame (CA par ville) : 0.82 s
